# System Visualizations Dashboard
Comprehensive visualization notebook for The Gatekeeper system with data analytics, metrics monitoring, network graphs, and interactive controls.

In [ ]:
# Import core libraries
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Rectangle, FancyBboxPatch
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Jupyter magic commands
%matplotlib inline
%load_ext autoreload
%autoreload 2

# Style configuration
plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

print("✓ Libraries loaded successfully")

## 1. Interactive Widgets & Controls
Create interactive parameter controls for system tuning.

In [ ]:
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Configuration controls
temperature_slider = widgets.FloatSlider(
    value=0.7,
    min=0.0,
    max=2.0,
    step=0.1,
    description='Temperature:',
    style={'description_width': 'initial'}
)

top_p_slider = widgets.FloatSlider(
    value=0.9,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Top-P:',
    style={'description_width': 'initial'}
)

max_tokens_slider = widgets.IntSlider(
    value=1024,
    min=128,
    max=4096,
    step=128,
    description='Max Tokens:',
    style={'description_width': 'initial'}
)

model_dropdown = widgets.Dropdown(
    options=['gpt-4', 'gpt-3.5-turbo', 'claude-3-opus', 'claude-3-sonnet', 'llama-2-70b'],
    value='gpt-4',
    description='Model:',
    style={'description_width': 'initial'}
)

def update_config(temperature, top_p, max_tokens, model):
    """Update and display configuration"""
    config = {
        'temperature': temperature,
        'top_p': top_p,
        'max_tokens': max_tokens,
        'model': model
    }
    print("Current Configuration:")
    print(json.dumps(config, indent=2))
    return config

# Create interactive widget
config_widget = interactive(
    update_config,
    temperature=temperature_slider,
    top_p=top_p_slider,
    max_tokens=max_tokens_slider,
    model=model_dropdown
)

display(config_widget)

## 2. System Metrics Dashboard
Monitor real-time performance metrics from Prometheus endpoints.

In [ ]:
def create_metrics_dashboard(metrics_data=None):
    """
    Create interactive metrics dashboard with Plotly
    
    Args:
        metrics_data: Dictionary with metric names and time series data
    """
    # Sample data if none provided
    if metrics_data is None:
        timestamps = pd.date_range(start='now', periods=100, freq='1s')
        metrics_data = {
            'cpu_usage': np.random.uniform(20, 80, 100),
            'memory_usage': np.random.uniform(40, 70, 100),
            'response_time': np.random.gamma(2, 50, 100),
            'requests_per_sec': np.random.poisson(25, 100),
            'error_rate': np.random.uniform(0, 5, 100),
            'active_connections': np.random.randint(10, 50, 100)
        }
        df = pd.DataFrame(metrics_data, index=timestamps)
    
    # Create subplots
    fig = make_subplots(
        rows=3, cols=2,
        subplot_titles=('CPU Usage (%)', 'Memory Usage (%)', 
                       'Response Time (ms)', 'Requests/sec',
                       'Error Rate (%)', 'Active Connections'),
        vertical_spacing=0.12,
        horizontal_spacing=0.1
    )
    
    # Add traces
    fig.add_trace(go.Scatter(y=df['cpu_usage'], mode='lines', 
                            line=dict(color='#00ffff', width=2),
                            fill='tozeroy', name='CPU'),
                 row=1, col=1)
    
    fig.add_trace(go.Scatter(y=df['memory_usage'], mode='lines',
                            line=dict(color='#ff00ff', width=2),
                            fill='tozeroy', name='Memory'),
                 row=1, col=2)
    
    fig.add_trace(go.Scatter(y=df['response_time'], mode='lines',
                            line=dict(color='#ffff00', width=2),
                            name='Response Time'),
                 row=2, col=1)
    
    fig.add_trace(go.Bar(y=df['requests_per_sec'],
                        marker=dict(color='#00ff00'),
                        name='Requests'),
                 row=2, col=2)
    
    fig.add_trace(go.Scatter(y=df['error_rate'], mode='lines+markers',
                            line=dict(color='#ff0000', width=2),
                            marker=dict(size=4),
                            name='Errors'),
                 row=3, col=1)
    
    fig.add_trace(go.Scatter(y=df['active_connections'], mode='lines',
                            line=dict(color='#ff8800', width=2),
                            name='Connections'),
                 row=3, col=2)
    
    # Update layout
    fig.update_layout(
        height=900,
        showlegend=False,
        template='plotly_dark',
        title_text="System Metrics Dashboard",
        title_font_size=20
    )
    
    return fig

# Display dashboard
metrics_fig = create_metrics_dashboard()
metrics_fig.show()

## 3. Network Graph Visualization
Visualize system architecture and component relationships.

In [ ]:
import networkx as nx

def create_system_network_graph():
    """
    Create interactive network graph of system components
    """
    # Create graph
    G = nx.Graph()
    
    # Define system components and relationships
    components = {
        'Core': ['Omega System', 'Gatekeeper', 'Orchestrator'],
        'AI': ['LLM Engine', 'TTS System', 'Speech Recognition', 'Vision'],
        'Data': ['Database', 'Cache', 'File Storage'],
        'Network': ['API Gateway', 'WebSocket Server', 'Load Balancer'],
        'Monitoring': ['Prometheus', 'Logging', 'Metrics Collector']
    }
    
    # Add nodes with categories
    for category, nodes in components.items():
        for node in nodes:
            G.add_node(node, category=category)
    
    # Add edges (relationships)
    edges = [
        ('Omega System', 'Orchestrator'),
        ('Gatekeeper', 'Orchestrator'),
        ('Orchestrator', 'LLM Engine'),
        ('Orchestrator', 'API Gateway'),
        ('LLM Engine', 'TTS System'),
        ('LLM Engine', 'Speech Recognition'),
        ('LLM Engine', 'Vision'),
        ('API Gateway', 'WebSocket Server'),
        ('API Gateway', 'Load Balancer'),
        ('Database', 'Cache'),
        ('Orchestrator', 'Database'),
        ('Prometheus', 'Metrics Collector'),
        ('Logging', 'Metrics Collector'),
        ('Orchestrator', 'Prometheus')
    ]
    G.add_edges_from(edges)
    
    # Calculate layout
    pos = nx.spring_layout(G, k=2, iterations=50)
    
    # Create edge trace
    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
    
    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=2, color='#888'),
        hoverinfo='none',
        mode='lines'
    )
    
    # Create node traces by category
    category_colors = {
        'Core': '#ff0000',
        'AI': '#00ff00',
        'Data': '#0000ff',
        'Network': '#ffff00',
        'Monitoring': '#ff00ff'
    }
    
    node_traces = []
    for category, color in category_colors.items():
        node_x = []
        node_y = []
        node_text = []
        
        for node in G.nodes():
            if G.nodes[node]['category'] == category:
                x, y = pos[node]
                node_x.append(x)
                node_y.append(y)
                node_text.append(f"{node}<br>Category: {category}<br>Connections: {len(list(G.neighbors(node)))}")
        
        node_trace = go.Scatter(
            x=node_x, y=node_y,
            mode='markers+text',
            hoverinfo='text',
            text=[t.split('<br>')[0] for t in node_text],
            hovertext=node_text,
            textposition="top center",
            name=category,
            marker=dict(
                size=30,
                color=color,
                line=dict(width=2, color='white')
            )
        )
        node_traces.append(node_trace)
    
    # Create figure
    fig = go.Figure(data=[edge_trace] + node_traces,
                   layout=go.Layout(
                       title='System Architecture Network Graph',
                       titlefont_size=20,
                       showlegend=True,
                       hovermode='closest',
                       template='plotly_dark',
                       height=700,
                       xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                       yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
                   ))
    
    return fig

# Display network graph
network_fig = create_system_network_graph()
network_fig.show()

## 4. Data Analytics & Statistical Plots
Analyze performance data with statistical visualizations.

In [ ]:
def create_analytics_dashboard():
    """
    Create comprehensive analytics dashboard with multiple chart types
    """
    # Generate sample performance data
    np.random.seed(42)
    dates = pd.date_range(start='2024-01-01', periods=365, freq='D')
    
    df = pd.DataFrame({
        'date': dates,
        'requests': np.random.poisson(1000, 365) + np.linspace(800, 1200, 365),
        'latency_p50': np.random.gamma(2, 30, 365),
        'latency_p95': np.random.gamma(3, 50, 365),
        'latency_p99': np.random.gamma(4, 70, 365),
        'success_rate': np.random.uniform(95, 99.9, 365),
        'active_users': np.random.randint(100, 500, 365)
    })
    
    # Create matplotlib figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('System Analytics Dashboard', fontsize=18, fontweight='bold')
    
    # 1. Request volume over time with trend
    ax1 = axes[0, 0]
    ax1.plot(df['date'], df['requests'], alpha=0.6, linewidth=1, label='Daily Requests')
    z = np.polyfit(range(len(df)), df['requests'], 2)
    p = np.poly1d(z)
    ax1.plot(df['date'], p(range(len(df))), 'r--', linewidth=2, label='Trend')
    ax1.fill_between(df['date'], df['requests'], alpha=0.3)
    ax1.set_title('Request Volume Trend', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Requests per Day')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # 2. Latency percentiles
    ax2 = axes[0, 1]
    ax2.plot(df['date'], df['latency_p50'], label='P50', linewidth=2, color='green')
    ax2.plot(df['date'], df['latency_p95'], label='P95', linewidth=2, color='orange')
    ax2.plot(df['date'], df['latency_p99'], label='P99', linewidth=2, color='red')
    ax2.fill_between(df['date'], df['latency_p50'], df['latency_p99'], alpha=0.2)
    ax2.set_title('Response Latency Percentiles', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Latency (ms)')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    # 3. Success rate distribution
    ax3 = axes[1, 0]
    ax3.hist(df['success_rate'], bins=30, color='cyan', alpha=0.7, edgecolor='white')
    ax3.axvline(df['success_rate'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["success_rate"].mean():.2f}%')
    ax3.axvline(df['success_rate'].median(), color='yellow', linestyle='--', linewidth=2, label=f'Median: {df["success_rate"].median():.2f}%')
    ax3.set_title('Success Rate Distribution', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Success Rate (%)')
    ax3.set_ylabel('Frequency')
    ax3.legend()
    ax3.grid(alpha=0.3)
    
    # 4. Correlation heatmap
    ax4 = axes[1, 1]
    corr_data = df[['requests', 'latency_p50', 'latency_p95', 'latency_p99', 'success_rate', 'active_users']].corr()
    im = ax4.imshow(corr_data, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
    ax4.set_xticks(range(len(corr_data.columns)))
    ax4.set_yticks(range(len(corr_data.columns)))
    ax4.set_xticklabels(corr_data.columns, rotation=45, ha='right')
    ax4.set_yticklabels(corr_data.columns)
    
    # Add correlation values
    for i in range(len(corr_data)):
        for j in range(len(corr_data)):
            text = ax4.text(j, i, f'{corr_data.iloc[i, j]:.2f}',
                          ha="center", va="center", color="black", fontsize=9, fontweight='bold')
    
    ax4.set_title('Metric Correlation Matrix', fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax4)
    
    plt.tight_layout()
    return fig, df

# Display analytics
analytics_fig, analytics_df = create_analytics_dashboard()
plt.show()

# Show summary statistics
print("\n=== Summary Statistics ===")
print(analytics_df.describe())

## 5. Real-Time Monitoring Integration
Connect to Prometheus metrics endpoint for live data.

In [ ]:
import requests
from typing import Dict, List

class MetricsCollector:
    """Collect metrics from Prometheus or custom endpoints"""
    
    def __init__(self, prometheus_url: str = 'http://localhost:9090'):
        self.prometheus_url = prometheus_url
        self.metrics_cache = {}
    
    def query_metric(self, query: str) -> Dict:
        """Query Prometheus for a specific metric"""
        try:
            response = requests.get(
                f"{self.prometheus_url}/api/v1/query",
                params={'query': query}
            )
            response.raise_for_status()
            return response.json()
        except Exception as e:
            print(f"Error querying metric: {e}")
            return {}
    
    def get_system_metrics(self) -> Dict:
        """Get common system metrics"""
        metrics = {}
        
        # Define common queries
        queries = {
            'cpu_usage': 'rate(process_cpu_seconds_total[5m]) * 100',
            'memory_usage': 'process_resident_memory_bytes / 1024 / 1024',
            'request_rate': 'rate(http_requests_total[1m])',
            'error_rate': 'rate(http_requests_total{status=~"5.."}[1m])',
            'response_time': 'histogram_quantile(0.95, rate(http_request_duration_seconds_bucket[5m]))'
        }
        
        for name, query in queries.items():
            result = self.query_metric(query)
            if result.get('data', {}).get('result'):
                metrics[name] = result['data']['result']
        
        return metrics
    
    def get_custom_metric(self, metric_name: str) -> Dict:
        """Get a custom metric by name"""
        return self.query_metric(metric_name)

# Example usage
collector = MetricsCollector()

# Interactive metric selector
metric_selector = widgets.Dropdown(
    options=['cpu_usage', 'memory_usage', 'request_rate', 'error_rate', 'response_time'],
    description='Metric:',
    style={'description_width': 'initial'}
)

refresh_button = widgets.Button(
    description='Refresh Metrics',
    button_style='success',
    icon='refresh'
)

output = widgets.Output()

def on_refresh_clicked(b):
    with output:
        clear_output()
        print(f"Fetching {metric_selector.value}...")
        # In production, this would fetch real metrics
        print("Note: Configure prometheus_url to fetch live metrics")
        print(f"Sample data for {metric_selector.value}:")
        print(json.dumps({'value': np.random.uniform(0, 100), 'timestamp': str(datetime.now())}, indent=2))

refresh_button.on_click(on_refresh_clicked)

display(widgets.VBox([metric_selector, refresh_button, output]))

## 6. Custom Visualization Functions
Utility functions for creating custom visualizations.

In [ ]:
def plot_time_series(data: pd.DataFrame, columns: List[str], title: str = "Time Series"):
    """
    Plot multiple time series on the same axes
    
    Args:
        data: DataFrame with datetime index
        columns: List of column names to plot
        title: Plot title
    """
    fig, ax = plt.subplots(figsize=(14, 6))
    
    for col in columns:
        ax.plot(data.index, data[col], label=col, linewidth=2, marker='o', markersize=3, alpha=0.7)
    
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.set_xlabel('Time', fontsize=12)
    ax.set_ylabel('Value', fontsize=12)
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3, linestyle='--')
    plt.xticks(rotation=45)
    plt.tight_layout()
    return fig

def plot_distribution_comparison(data: Dict[str, List], title: str = "Distribution Comparison"):
    """
    Plot violin plots comparing distributions
    
    Args:
        data: Dictionary of {label: values}
        title: Plot title
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    positions = range(1, len(data) + 1)
    parts = ax.violinplot(list(data.values()), positions=positions, showmeans=True, showmedians=True)
    
    # Customize colors
    colors = plt.cm.viridis(np.linspace(0, 1, len(data)))
    for pc, color in zip(parts['bodies'], colors):
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.set_xticks(positions)
    ax.set_xticklabels(list(data.keys()), rotation=45, ha='right')
    ax.set_ylabel('Value', fontsize=12)
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    return fig

def plot_heatmap(data: pd.DataFrame, title: str = "Heatmap", cmap: str = 'viridis'):
    """
    Create annotated heatmap
    
    Args:
        data: DataFrame to visualize
        title: Plot title
        cmap: Colormap name
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(data, cmap=cmap, aspect='auto')
    
    # Set ticks and labels
    ax.set_xticks(np.arange(len(data.columns)))
    ax.set_yticks(np.arange(len(data.index)))
    ax.set_xticklabels(data.columns)
    ax.set_yticklabels(data.index)
    
    # Rotate labels
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    
    # Annotate cells
    for i in range(len(data.index)):
        for j in range(len(data.columns)):
            text = ax.text(j, i, f"{data.iloc[i, j]:.2f}",
                          ha="center", va="center", color="white", fontsize=9)
    
    ax.set_title(title, fontsize=16, fontweight='bold')
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    return fig

# Example usage
print("✓ Custom visualization functions loaded")
print("\nAvailable functions:")
print("  - plot_time_series(data, columns, title)")
print("  - plot_distribution_comparison(data, title)")
print("  - plot_heatmap(data, title, cmap)")

## 7. Export & Reporting
Save visualizations and generate reports.

In [ ]:
from pathlib import Path
from datetime import datetime

class ReportGenerator:
    """Generate and export visualization reports"""
    
    def __init__(self, output_dir: str = './reports'):
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    def save_figure(self, fig, name: str, formats: List[str] = ['png', 'pdf']):
        """Save matplotlib figure in multiple formats"""
        for fmt in formats:
            filepath = self.output_dir / f"{name}_{self.timestamp}.{fmt}"
            fig.savefig(filepath, dpi=300, bbox_inches='tight')
            print(f"✓ Saved: {filepath}")
    
    def save_plotly(self, fig, name: str, formats: List[str] = ['html', 'png']):
        """Save plotly figure"""
        for fmt in formats:
            filepath = self.output_dir / f"{name}_{self.timestamp}.{fmt}"
            if fmt == 'html':
                fig.write_html(str(filepath))
            else:
                fig.write_image(str(filepath))
            print(f"✓ Saved: {filepath}")
    
    def export_data(self, data: pd.DataFrame, name: str, formats: List[str] = ['csv', 'json']):
        """Export data in multiple formats"""
        for fmt in formats:
            filepath = self.output_dir / f"{name}_{self.timestamp}.{fmt}"
            if fmt == 'csv':
                data.to_csv(filepath)
            elif fmt == 'json':
                data.to_json(filepath, orient='records', indent=2)
            elif fmt == 'excel':
                data.to_excel(filepath)
            print(f"✓ Saved: {filepath}")

# Initialize report generator
report_gen = ReportGenerator()

# Export button
export_button = widgets.Button(
    description='Export All Visualizations',
    button_style='info',
    icon='download'
)

export_output = widgets.Output()

def on_export_clicked(b):
    with export_output:
        clear_output()
        print("Exporting visualizations...")
        report_gen.save_figure(analytics_fig, 'analytics_dashboard')
        report_gen.save_plotly(metrics_fig, 'metrics_dashboard')
        report_gen.save_plotly(network_fig, 'network_graph')
        report_gen.export_data(analytics_df, 'analytics_data')
        print("\n✓ Export complete!")

export_button.on_click(on_export_clicked)

display(widgets.VBox([export_button, export_output]))

## 8. Quick Reference
Common tasks and code snippets.

In [ ]:
# Quick reference guide
reference = """
╔═══════════════════════════════════════════════════════════════╗
║          SYSTEM VISUALIZATIONS - QUICK REFERENCE              ║
╚═══════════════════════════════════════════════════════════════╝

📊 CREATE METRICS DASHBOARD:
   fig = create_metrics_dashboard(metrics_data)
   fig.show()

🕸️  CREATE NETWORK GRAPH:
   fig = create_system_network_graph()
   fig.show()

📈 PLOT TIME SERIES:
   plot_time_series(df, ['col1', 'col2'], title="My Plot")

🔥 CREATE HEATMAP:
   plot_heatmap(df, title="Correlation", cmap='coolwarm')

💾 EXPORT VISUALIZATIONS:
   report_gen.save_figure(fig, 'name', formats=['png', 'pdf'])
   report_gen.export_data(df, 'data', formats=['csv', 'json'])

🔄 FETCH METRICS:
   collector = MetricsCollector('http://localhost:9090')
   metrics = collector.get_system_metrics()

🎛️  INTERACTIVE CONTROLS:
   Use the widget controls above to adjust parameters
   in real-time and see immediate visual feedback

📦 REQUIRED PACKAGES:
   matplotlib, plotly, pandas, numpy, networkx,
   ipywidgets, prometheus-client, requests

═══════════════════════════════════════════════════════════════
"""

print(reference)